# Proyecto Final — Programacion II | MCD CUCEA 2026-A
## Machine Learning: Algoritmo Supervisado y No Supervisado
**Autor:** Carlos Pulido Rosas  
**Tema de tesis:** Riesgo de Automatizacion Laboral en Jalisco  
**Datos:** Vista `vista_riesgo_jalisco` en SQL Server (Defunciones_2000)  

---

### Pipeline
```
SQL Server (vista_riesgo_jalisco)
  → Regresion Lineal Multiple  (supervisado  — U3)
  → K-Means con Elbow Method   (no supervisado — U4)
```

> **Pregunta de tesis:** ¿El perfil sociolaboral del trabajador (educacion,
> ingreso, edad, sector, formalidad) predice su riesgo de automatizacion?

## 0. Importaciones y conexion a SQL Server

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sqlalchemy import create_engine
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

np.random.seed(42)

# Conexion via SQLAlchemy (Windows Authentication)
engine = create_engine(
    'mssql+pyodbc://localhost,1433/Defunciones_2000'
    '?driver=ODBC+Driver+17+for+SQL+Server'
    '&trusted_connection=yes'
    '&TrustServerCertificate=yes'
)

print('Motor SQLAlchemy creado exitosamente')
print('Servidor: localhost:1433 | BD: Defunciones_2000')

## 1. Carga de datos desde la VIEW analitica

La vista `vista_riesgo_jalisco` consolida el perfil ENOE del trabajador
con los descriptores O*NET. Aqui cargamos todo el dataset.

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql('SELECT * FROM dbo.vista_riesgo_jalisco', conn)

print(f'Registros cargados  : {len(df):,}')
print(f'Columnas            : {list(df.columns)}')
print(f'\nDistribucion de riesgo_categoria:')
print(df['riesgo_categoria'].value_counts())
df.head()

## 2. Preprocesamiento

In [ ]:
# Verificar nulos
print('Valores nulos:', df.isnull().sum().sum())

# Imputar nulos numericos con mediana
num_cols = ['nivel_educativo', 'ingreso_mensual', 'edad', 'tamanio_empresa',
            'frey_osborne_score']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Codificar variables categoricas con LabelEncoder
le_sector    = LabelEncoder()
le_formalidad = LabelEncoder()
df['sector_enc']    = le_sector.fit_transform(df['sector'])
df['formalidad_enc'] = le_formalidad.fit_transform(df['formalidad'])

print('Sectores codificados:', dict(zip(le_sector.classes_,
                                       le_sector.transform(le_sector.classes_))))
print('Formalidad codificada:', dict(zip(le_formalidad.classes_,
                                         le_formalidad.transform(le_formalidad.classes_))))

---
## 3. Algoritmo Supervisado — Regresion Lineal Multiple

**Justificacion:** La variable objetivo `frey_osborne_score` es continua (0–1),
lo que hace de la **Regresion Lineal Multiple** el algoritmo apropiado de U3.

**Features elegidos — perfil ENOE + IRA (sociolaboral + incentivo economico):**
Se excluyen los descriptores O*NET (rti, cognitive_demand, etc.) para evitar
data leakage. Se incluye el IRA como variable economica derivada de Censos INEGI.

| Feature | Fuente | Descripcion |
|---------|--------|-------------|
| nivel_educativo | ENOE | Escolaridad acumulada (1–6) |
| ingreso_mensual | ENOE | Ingreso mensual en MXN |
| edad | ENOE | Edad del trabajador |
| tamanio_empresa | ENOE | Tamano del establecimiento |
| sector_enc | ENOE | Sector economico SCIAN (codificado) |
| formalidad_enc | ENOE | Formal=1 / Informal=0 |
| ira | ENOE + Censos INEGI 2019 | Indice de Rentabilidad de la Automatizacion (ingocup_anual / activo_fijo_amortizado) |

> **IRA (Acemoglu & Restrepo, 2018):** IRA > 1 indica que el salario anual supera
> el costo de automatizar — existe incentivo economico real para sustituir al trabajador.

In [ ]:
# Features: perfil sociolaboral ENOE + IRA (incentivo economico)
features = ['nivel_educativo', 'ingreso_mensual', 'edad',
            'tamanio_empresa', 'sector_enc', 'formalidad_enc', 'ira']

X = df[features]
y = df['frey_osborne_score']

# Division 70/30 con random_state fijo (reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f'Entrenamiento : {len(X_train):,} registros ({len(X_train)/len(X):.0%})')
print(f'Prueba        : {len(X_test):,} registros ({len(X_test)/len(X):.0%})')
print(f'Features      : {features}')

In [ ]:
# Estandarizacion — obligatoria para interpretar coeficientes comparativamente
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Entrenar modelo
modelo_lr = LinearRegression()
modelo_lr.fit(X_train_sc, y_train)

# Prediccion y metricas
y_pred = modelo_lr.predict(X_test_sc)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R² Score : {r2:.4f}  — varianza explicada por variables ENOE')
print(f'RMSE     : {rmse:.4f} — error promedio en escala 0–1 del score')
print()
print('Interpretacion: el perfil sociolaboral (educacion, ingreso, sector)')
print(f'explica el {r2*100:.1f}% de la varianza del riesgo de automatizacion.')
print('El R² restante corresponde a factores especificos de la tarea (O*NET).')

In [ ]:
# Coeficientes estandarizados — importancia relativa de variables
coef_df = pd.DataFrame({
    'Variable': features,
    'Coeficiente': modelo_lr.coef_
}).sort_values('Coeficiente', key=abs, ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Regresion Lineal Multiple — Riesgo de Automatizacion en Jalisco',
             fontsize=13, fontweight='bold')

# Coeficientes
colors = ['#E53935' if c > 0 else '#1E88E5' for c in coef_df['Coeficiente']]
axes[0].barh(coef_df['Variable'], coef_df['Coeficiente'], color=colors)
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Coeficientes estandarizados\n(rojo=aumenta riesgo | azul=reduce riesgo)', fontsize=10)
axes[0].set_xlabel('Coeficiente')

# Predicciones vs valores reales
axes[1].scatter(y_test, y_pred, alpha=0.5, color='#1E88E5', s=20)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Prediccion perfecta')
axes[1].set_xlabel('Valor real (frey_osborne_score)')
axes[1].set_ylabel('Valor predicho')
axes[1].set_title(f'Predicciones vs Valores Reales\nR²={r2:.3f} | RMSE={rmse:.3f}', fontsize=10)
axes[1].legend()

plt.tight_layout()
plt.savefig('regresion_lineal_resultados.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafica guardada: regresion_lineal_resultados.png')

In [ ]:
# Curva de aprendizaje — detectar overfitting/underfitting
train_sizes, train_scores, test_scores = learning_curve(
    LinearRegression(), X_train_sc, y_train,
    cv=5, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 8)
)

plt.figure(figsize=(8, 4))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Entrenamiento', color='#E53935')
plt.plot(train_sizes, test_scores.mean(axis=1),  'o-', label='Validacion (CV-5)', color='#1E88E5')
plt.fill_between(train_sizes,
                 train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='#E53935')
plt.fill_between(train_sizes,
                 test_scores.mean(axis=1) - test_scores.std(axis=1),
                 test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.1, color='#1E88E5')
plt.xlabel('Tamano del conjunto de entrenamiento')
plt.ylabel('R²')
plt.title('Curva de Aprendizaje — Regresion Lineal Multiple')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('curva_aprendizaje.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafica guardada: curva_aprendizaje.png')

### Interpretacion — Algoritmo Supervisado

La Regresion Lineal Multiple con **solo variables ENOE** (perfil sociolaboral)
permite cuantificar cuanto explica el entorno del trabajador —
independientemente de la descripcion especifica de su tarea:

- **nivel_educativo** tiene el coeficiente negativo mas fuerte: mayor escolaridad
  reduce el riesgo de automatizacion (factor protector).
- **formalidad_enc** (Formal=1) reduce el riesgo: los trabajadores formales
  tienden a ocupaciones con mayor demanda cognitiva.
- **ingreso_mensual** tiene coeficiente negativo: mayores ingresos correlacionan
  con menor riesgo de automatizacion.
- La brecha entre R² de entrenamiento y validacion es pequena,
  indicando ausencia de overfitting.

El R² residual justifica agregar los descriptores O*NET y variables de
exposicion LLM en la Fase 2 de la tesis (Bloques 3 y 4).

In [ ]:
# IRA por sector: distribucion y relacion con riesgo tecnico
ira_sector = df.groupby('sector').agg(
    ira_prom=('ira', 'mean'),
    riesgo_prom=('frey_osborne_score', 'mean'),
    n=('id_trabajador', 'count')
).sort_values('ira_prom', ascending=True).round(3)

print('IRA promedio por sector:')
print(ira_sector)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('IRA — Indice de Rentabilidad de la Automatizacion por Sector',
             fontsize=13, fontweight='bold')

# IRA por sector
colors_bar = ['#E53935' if v > 1 else '#43A047' for v in ira_sector['ira_prom']]
axes[0].barh(ira_sector.index, ira_sector['ira_prom'], color=colors_bar)
axes[0].axvline(x=1, color='black', linewidth=1.5, linestyle='--', label='IRA = 1 (umbral)')
axes[0].set_xlabel('IRA promedio (ingocup_anual / activo_fijo_amortizado)')
axes[0].set_title('IRA por Sector\n(rojo = incentivo economico de automatizar)')
axes[0].legend()

# IRA vs riesgo tecnico: la paradoja de la agricultura
scatter = axes[1].scatter(
    ira_sector['ira_prom'],
    ira_sector['riesgo_prom'],
    s=ira_sector['n'] * 0.5,
    alpha=0.7, color='#1E88E5', edgecolors='white'
)
for sector, row in ira_sector.iterrows():
    axes[1].annotate(sector, (row['ira_prom'], row['riesgo_prom']),
                     fontsize=7, ha='left', va='bottom')
axes[1].axvline(x=1, color='gray', linewidth=1, linestyle='--')
axes[1].axhline(y=0.5, color='gray', linewidth=1, linestyle='--')
axes[1].set_xlabel('IRA promedio')
axes[1].set_ylabel('Riesgo Frey-Osborne promedio')
axes[1].set_title('IRA vs Riesgo Tecnico por Sector\n(tamano = n trabajadores)')

plt.tight_layout()
plt.savefig('ira_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafica guardada: ira_analisis.png')
print()
print('Correlacion IRA <-> frey_osborne_score:', round(df['ira'].corr(df['frey_osborne_score']), 4))

---
## 3.5 Analisis del IRA — Incentivo Economico de Automatizacion

El **IRA** (Indice de Rentabilidad de la Automatizacion) operacionaliza la
hipotesis de Acemoglu & Restrepo (2018): la automatizacion ocurre cuando
el costo salarial supera el costo de la tecnologia sustituta.

> IRA = ingocup_anual / activo_fijo_amortizado_5_anios (por sector SCIAN)
>
> - IRA > 1 → el salario anual supera el costo de automatizar → **incentivo economico real**
> - IRA < 1 → automatizar cuesta mas que mantener al trabajador → **sin incentivo economico**

**Hipotesis H2 de tesis:** El IRA modera la relacion entre riesgo tecnico y
adopcion real. Explica por que agricultura (alto riesgo Frey-Osborne) sigue
siendo manual en Jalisco pese al riesgo teorico elevado.

---
## 4. Algoritmo No Supervisado — K-Means

**Justificacion:** K-Means agrupa ocupaciones por su perfil completo de riesgo
(variables O*NET + sociolaborales) sin etiquetas previas.
Se usa porque los datos son numericos continuos y se busca identificar
grupos naturales accionables para politica publica (U4 del programa).

**Pregunta de tesis:** ¿Cuantos grupos diferenciados de ocupaciones existen
segun su perfil de automatizacion en Jalisco?

In [ ]:
# Features para clustering: perfil completo (ENOE + O*NET)
cluster_features = ['frey_osborne_score', 'rti', 'cognitive_demand',
                    'social_interaction', 'creativity',
                    'nivel_educativo', 'ingreso_mensual']

X_cluster = df[cluster_features].copy()

# Escalar — obligatorio para K-Means (sensible a magnitud)
scaler_km = StandardScaler()
X_scaled = scaler_km.fit_transform(X_cluster)

# Diagrama de Codo — determinar k optimo
inercias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inercias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', linewidth=1.5, label='k=3 (optimo)')
plt.xlabel('Numero de clusters (k)')
plt.ylabel('Inercia (WCSS)')
plt.title('Diagrama de Codo — Determinacion del k Optimo')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafica guardada: kmeans_elbow.png')

In [ ]:
# K-Means final con k=3
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_scaled)

# Ordenar clusters por riesgo promedio para asignar etiquetas consistentes
riesgo_por_cluster = df.groupby('cluster')['frey_osborne_score'].mean().sort_values()
mapa_cluster = {riesgo_por_cluster.index[0]: 'Riesgo Bajo',
                riesgo_por_cluster.index[1]: 'Riesgo Medio',
                riesgo_por_cluster.index[2]: 'Riesgo Alto'}
df['cluster_label'] = df['cluster'].map(mapa_cluster)

# Resumen por cluster
resumen = df.groupby('cluster_label').agg(
    n_trabajadores=('id_trabajador', 'count'),
    riesgo_prom=('frey_osborne_score', 'mean'),
    rti_prom=('rti', 'mean'),
    cognitivo_prom=('cognitive_demand', 'mean'),
    educacion_prom=('nivel_educativo', 'mean'),
    ingreso_prom=('ingreso_mensual', 'mean')
).round(3)

print('Perfil de clusters:')
print(resumen)

In [ ]:
# PCA a 2 componentes para visualizacion
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
varianza = pca.explained_variance_ratio_.sum()

print(f'Varianza explicada por 2 componentes PCA: {varianza:.1%}')

colores = {'Riesgo Alto': '#E53935', 'Riesgo Medio': '#FDD835', 'Riesgo Bajo': '#43A047'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('K-Means (k=3) — Segmentacion de Ocupaciones por Riesgo en Jalisco',
             fontsize=13, fontweight='bold')

# Proyeccion PCA
for label, color in colores.items():
    mask = df['cluster_label'] == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=label, alpha=0.6, s=25, edgecolors='none')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].set_title(f'Proyeccion PCA (varianza: {varianza:.1%})')
axes[0].legend()

# Riesgo promedio por cluster
riesgo_mean = df.groupby('cluster_label')['frey_osborne_score'].mean().reindex(
    ['Riesgo Bajo', 'Riesgo Medio', 'Riesgo Alto'])
bar_colors = [colores[l] for l in riesgo_mean.index]
axes[1].bar(riesgo_mean.index, riesgo_mean.values, color=bar_colors, edgecolor='white')
axes[1].set_ylabel('Frey-Osborne Score promedio')
axes[1].set_title('Riesgo Promedio por Cluster')
axes[1].set_ylim(0, 1)
for i, v in enumerate(riesgo_mean.values):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('kmeans_clusters_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafica guardada: kmeans_clusters_pca.png')

### Interpretacion — Algoritmo No Supervisado

El **Diagrama de Codo** confirma k=3 como el numero optimo de clusters,
coincidiendo con la clasificacion teorica de Frey & Osborne.

| Cluster | Perfil | Ocupaciones representativas |
|---------|--------|------------------------------|
| Riesgo Alto | RTI alto, educacion baja, ingreso bajo | Jornaleros, operadores de maquinaria, cajeros |
| Riesgo Medio | RTI medio, perfil administrativo | Tecnicos, vendedores, administrativos |
| Riesgo Bajo | Cognitivo alto, educacion alta | Profesionistas, docentes, directivos |

**Hallazgo critico:** El cluster de Riesgo Medio (administrativos/tecnicos)
es el mas numeroso en Jalisco y el mas expuesto a LLMs — los modelos de
lenguaje realizan exactamente las tareas que Frey-Osborne clasificaba
como 'seguras'. Este hallazgo motiva la Fase 2 de la tesis.

---
## 5. Discusion integrada

| Etapa | Pregunta respondida | Hallazgo clave |
|-------|--------------------|-----------------|
| SQL — INNER JOIN | Cobertura de datos | 82% de ocupaciones mapeadas |
| SQL — LEFT JOIN | Brechas de cobertura | 18% sin O*NET son los mas vulnerables |
| SQL — GROUP BY | Riesgo por sector | Manufactura y Agricultura lideran |
| Orange Flujo 1 | Distribucion | Riesgo bimodal; informalidad amplifica |
| Orange Flujo 2 | Correlaciones | cognitive_demand r=−0.83 con riesgo |
| Orange Flujo 3 | Modelo visual | RF Accuracy=0.84; AUC=0.91 |
| Regresion Lineal | Cuantificacion | Educacion y formalidad: predictores negativos |
| K-Means | Segmentacion | 3 grupos accionables; cluster medio = nueva zona de riesgo |

Los resultados convergen: el riesgo de automatizacion en Jalisco esta
determinado por la naturaleza cognitiva de las tareas, el nivel educativo
y la formalidad laboral. El R² de la regresion lineal (solo ENOE) cuantifica
cuanto del riesgo es predecible desde el perfil observable del trabajador,
y la brecha restante justifica agregar los bloques LLM e IRA en la Fase 2.